# 03 — Graph / ν export + conservation layer

Visualizes the pipeline node that builds the stoichiometric export
(`data/stoich/nu_mesa{80,151}.npz`), the constraint matrix C and Target-B
projector, and the conservation gate that must pass **before any training**
(root CLAUDE.md invariant #1). Closes phase-0 checklist rows 1 (reaction
counts / graph extents), 7 (cond(ν)), and 10 (projector drift).

Exploratory only — citable numbers live in RESULTS.md; this notebook
re-plots the exported artifacts and mirrors the gate's arithmetic for
illustration.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs

nbs.style()
QUICK = nbs.QUICK

In [ ]:
nbs.provenance_header(
    "03",
    "Graph / ν export + conservation layer",
    nbs.status_of([1, 7, 10]),
    results_rows=[
        "2026-07-08: graph export + conservation layer (drift ≤ 1e-12 gate, cond(ν), graph extents)",
        "2026-07-09: MESA-reconciled reaction counts — mesa_80 607 / mesa_151 1518 (ADR 0003)",
        "2026-07-08: projector residual ≤ 1e-16 across dY scales 1e0…1e-20 (checklist row 10)",
    ],
    data=["data/stoich/nu_mesa{80,151}.npz", "data/graphs/mesa{80,151}.graphml"],
    scripts=[
        "scripts/export_stoich_matrix.py",
        "scripts/graph_metrics.py",
        "scripts/check_conservation.py",
        "scripts/check_projector.py",
    ],
)

## Load the exported stoichiometric artifacts

`np.load` on the canonical npz — deliberately NOT `gnn_nucleo.graph`
(whose package import pulls pynucastro + networkx; the npz IS the exported
artifact the training stack consumes). Everything below is float64.

In [ ]:
NETS = ["mesa_80", "mesa_151"]
stoich = {net: nbs.load_nu(net) for net in NETS}
for net, z in stoich.items():
    print(
        f"{net}: nu {z['nu'].shape}  nu_ext {z['nu_ext'].shape}  C {z['C'].shape}  "
        f"weak columns {int(z['weak_mask'].sum())}  dtype {z['nu'].dtype}"
    )

## Figure 1 — ν sparsity structure

Species rows sorted by mass number A; columns are reactions in export order.
The dense light-isotope band at the top (n, p, α participate everywhere) and
the block-diagonal-ish heavy structure are what the GNN's bipartite message
passing traverses.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), height_ratios=[80, 151])
for ax, net in zip(axes, NETS):
    z = stoich[net]
    order = np.argsort(z["A"], kind="stable")
    ax.spy(z["nu"][order], aspect="auto", markersize=0.6, color="#0072B2")
    ax.set_ylabel(f"{net}\nspecies (by A)")
    ax.grid(False)
axes[0].set_title("ν sparsity — nonzero stoichiometric coefficients (species × reaction)")
axes[1].set_xlabel("reaction column (export order)")
nbs.caption(
    fig,
    "Stoichiometric matrices as exported: mesa_80 80×607, mesa_151 151×1518 — the "
    "MESA-reconciled reaction sets (MESA_ONLY = 0, post-drop counts equal MESA exactly). "
    "These column counts ARE the Target-A flux-head output dimensions.",
    results=["RESULTS.md 2026-07-09 reconciliation rows (checklist row 1; ADR 0003)"],
    scripts=["scripts/export_stoich_matrix.py", "scripts/reconcile_reactions.py"],
)

## Figure 2 — the conservation gate: drift vs φ magnitude

Invariant #1: dY = νφ must conserve baryon number and charge-to-lepton
closure to ≤ 1e-12 **for ANY φ, including random ones** — conservation is a
property of the ν export, not of training. Mirrors
`tests/test_conservation.py`: random φ across 20 decades of magnitude, drift
bounded by max(1e-12·s, 1e-13·G) with s = min(1, max|φ|) and
G = |A|·(|ν|·|φ|) (relative-to-gross at the small end, where an absolute
bound is vacuous).

In [ ]:
rng = np.random.default_rng(0)
scales = np.logspace(-20, 0, 5 if QUICK else 21)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, net in zip(axes, NETS):
    z = stoich[net]
    nu_ext, C, A = z["nu_ext"], z["C"], z["A"]
    absnu = np.abs(z["nu"])
    drift, bound = [], []
    for s in scales:
        phi = rng.standard_normal(nu_ext.shape[1]) * s
        dY_ext = nu_ext @ phi  # float64 throughout
        resid = np.abs(C @ dY_ext)  # rows: baryon, charge(+e⁻), lepton
        gross = float(np.abs(A) @ (absnu @ np.abs(phi)))
        drift.append(resid.max())
        bound.append(max(1e-12 * min(1.0, np.abs(phi).max()), 1e-13 * gross))
    ax.loglog(scales, np.maximum(drift, 1e-30), "o-", label="measured max drift")
    ax.loglog(scales, bound, "--", color="#D55E00", label="gate bound")
    ax.set_title(f"{net}")
    ax.set_xlabel("φ scale (max |φ|)")
axes[0].set_ylabel("max |C · νφ| (baryon / charge / lepton rows)")
axes[0].legend()
fig.suptitle("Conservation drift for RANDOM φ vs the gate bound (float64)", y=1.02)
nbs.caption(
    fig,
    "Drift sits at float64 rounding, orders below the bound, at every φ scale — with an "
    "untrained random flux head, exactly as the gate requires before training may start. "
    "Column drifts were measured exactly 0.0. Quick-look re-roll of the test's arithmetic; "
    "the gate itself is tests/test_conservation.py.",
    results=[
        "RESULTS.md 2026-07-08 graph-export/conservation rows (gate ≤ 1e-12; scale-swept bound)"
    ],
    scripts=["tests/test_conservation.py", "scripts/check_conservation.py"],
)

## Figure 3 — Target-B projector residual across 20 decades of dY

The fallback target (Target B) projects raw dY onto null(C):
P = I − Cᵀ(CCᵀ)⁻¹C, valid ONLY in a linear output space (the documented
NuGNN signed-log failure mode). Checklist row 10 measured max relative
residual ≤ 1e-16 across dY scales 1e0…1e-20 — static-operator viability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for net in NETS:
    C = stoich[net]["C"]
    n_ext = C.shape[1]
    P = np.eye(n_ext) - C.T @ np.linalg.solve(C @ C.T, C)
    resid = []
    for s in scales:
        dY = rng.standard_normal(n_ext) * s
        r = np.abs(C @ (P @ dY)).max() / max(np.abs(C @ dY).max(), 1e-300)
        resid.append(r)
    ax.loglog(scales, resid, "o-", label=net)
ax.axhline(1e-12, color="#D55E00", ls="--", label="gate ≤ 1e-12")
ax.set_xlabel("dY scale")
ax.set_ylabel("constraint residual, relative\nmax|C·P dY| / max|C·dY|")
ax.set_title("Null-space projector residual vs dY dynamic range (float64)")
ax.legend()
nbs.caption(
    fig,
    "Projection kills the constraint violation to float64 rounding (~1e-16 relative) at "
    "every scale, 4+ orders inside the 1e-12 gate; row 10's measurement additionally "
    "verified weak dYₑ preserved to ≤ 3.4e-21. Training-stability half of Target-B "
    "viability stayed with the kill-test (verdict: Target A, no switch).",
    results=["RESULTS.md 2026-07-08 projector rows (checklist row 10)"],
    scripts=["scripts/check_projector.py"],
)

## Figure 4 — bipartite graph extents ⇒ message-passing depth K

The exported GraphML is the isotope↔reaction bipartite digraph. Its radius
sets the processor depth: K ≈ ⌈radius⌉ + 2 = 5 (K = 4 with isotope→isotope
edges). Condition numbers of ν (full) are the row-7 quantities.

In [ ]:
import networkx as nx  # cheap; pynucastro NOT imported

from gnn_nucleo.killtest.active_set import cond_s_active  # rank-revealing cond (cheap import)

ext = {}
for net in NETS:
    g = nx.read_graphml(nbs.graphml_path(net)).to_undirected()
    ecc = nx.eccentricity(g)  # exact BFS from every node; ≤1669 nodes, fine
    ext[net] = dict(
        n_nodes=g.number_of_nodes(),
        n_edges=g.number_of_edges(),
        radius=min(ecc.values()),
        diameter=max(ecc.values()),
        # RANK-REVEALING cond (nonzero singular values only), the same definition as
        # graph/metrics.condition_numbers: ν has an exact structural left-null vector
        # (A·ν = 0, nullity 1), so np.linalg.cond would return ~1e16 — the null space is
        # the conservation law, not ill-conditioning.
        cond_nu=cond_s_active(stoich[net]["nu"], np.ones(stoich[net]["nu"].shape[1], bool)),
    )
    print(net, ext[net])

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(NETS))
ax.bar(x - 0.18, [ext[n]["radius"] for n in NETS], 0.36, label="radius")
ax.bar(x + 0.18, [ext[n]["diameter"] for n in NETS], 0.36, label="diameter")
for i, net in enumerate(NETS):
    ax.annotate(
        f"K = {ext[net]['radius'] + 2}\ncond(ν) = {ext[net]['cond_nu']:.1f}",
        (i, ext[net]["diameter"] + 0.15),
        ha="center",
        fontsize=9,
    )
ax.set_xticks(x, NETS)
ax.set_ylim(0, 8)
ax.set_ylabel("hops (bipartite graph)")
ax.set_title("Graph extents ⇒ processor depth K = radius + 2")
ax.legend()
nbs.caption(
    fig,
    "Radius 3 / diameter 6 on both networks ⇒ K = 5 message-passing steps (K = 4 with "
    "isotope→isotope edges); cond(ν) ≈ 42/58 (rank-revealing, nullity 1 — the null vector IS "
    "the baryon conservation law A·ν = 0, so it is excluded by construction rather than being "
    "ill-conditioning) — nowhere near the 1e6 Target-A→B switch line even before the kill-test "
    "active-set measurement.",
    results=[
        "RESULTS.md 2026-07-08 graph-metrics rows (checklist rows 1, 7)",
        "RESULTS.md 2026-07-09 extents unchanged post-reconciliation (ADR 0003)",
    ],
    scripts=["scripts/graph_metrics.py"],
)

## What this notebook does NOT show

- The **active-set** condition number cond(S_active) — that is the Step-6
  kill-test quantity (notebook 10; measured 41.8/57.4, mask empty).
- Rate VALUES: ν only encodes stoichiometry; the rate reconciliation and
  κ-floor story is notebook 04.
- The lepton-ledger construction details — `src/gnn_nucleo/graph/stoich.py`
  (conservation-critical; edits there auto-trigger the gate hook).